In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Check all api key's
try:
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    ELEVENLABS_API_KEY = os.environ["ELEVENLABS_API_KEY"]
    SPOONACULAR_API_KEY = os.environ["SPOONACULAR_API_KEY"]
    print("Loading API keys done.")
except KeyError as e:
    raise EnvironmentError(f"Missing api key: {e}")

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

Loading API keys done.


In [3]:
from sqlalchemy import create_engine, Column, Integer, String, Float, Text
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime

# Define the database class  
Base = declarative_base()

# User Profile Table
class UserProfile(Base):
    __tablename__ = 'user_profile'

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    weight = Column(Float)  # in kg
    height = Column(Float)  # in cm
    age = Column(Integer)   # 'Male' / 'Female'
    gender = Column(String)
    activity_level = Column(String)         # e.g., 'sedentary', 'active'
    goal = Column(String)              # e.g., 'weight_loss', 'muscle_gain'
    dietary_restrictions = Column(String)   # e.g., 'gluten, lactose' (comma separated)

# Meal History Table (for future tracking)
class MealLog(Base):
    __tablename__ = 'meal_log'

    id = Column(Integer, primary_key=True)
    date = Column(String, default=lambda: datetime.now().isoformat())
    recipe_name = Column(String)
    calories = Column(Integer)
    protein = Column(Integer)
    carbs = Column(Integer)
    fat = Column(Integer)
    user_rating = Column(Integer)   # rating 1-10
    user_notes = Column(Text, nullable=True)

# Create nowaste.db file
engine = create_engine('sqlite:///nowaste.db')
Base.metadata.create_all(engine)

# Create session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()

print("Database ready")

Database ready


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Widget Definitions ---
w_name = widgets.Text(description="Name:")
w_weight = widgets.FloatText(description="Weight (kg):", value=70.0)
w_height = widgets.FloatText(description="Height (cm):", value=175.0)
w_age = widgets.IntText(description="Age:", value=30)
w_gender = widgets.Dropdown(options=['Male', 'Female'], description="Gender:", )

# Activity levels mapped to descriptions
w_activity = widgets.Dropdown(
    options=[
        ('Sedentary (Office job, no sports)', 'sedentary'),
        ('Moderate (Exercise 1-3x/week)', 'moderate'),
        ('Active (Exercise 4-5x/week)', 'active'),
        ('Very Active (Athlete)', 'very_active')
    ],
    description="Activity:"
)

# Goals
w_goal = widgets.Dropdown(
    options=[
        ('Weight Loss (Reduction)', 'weight_loss'),
        ('Maintain Weight', 'maintenance'),
        ('Muscle Gain (Bulk)', 'muscle_gain')
    ],
    description="Goal:"
)

w_restrictions = widgets.Textarea(
    description="Intolerances:",
    placeholder="e.g. gluten, peanuts (leave empty if none)",
    
)

btn_save = widgets.Button(description="Save Profile", button_style='success')
output = widgets.Output()

# --- Callback Function to Save Data ---
def save_profile_to_db(b):
    with output:
        clear_output()
        
        # Check if a user already exists (assuming single-user app for now)
        existing_user = session.query(UserProfile).first()
        
        if not existing_user:
            existing_user = UserProfile()
            session.add(existing_user)
            print("Creating new profile...")
        else:
            print("Updating existing profile...")
        
        # Assign values from widgets to the database object
        existing_user.name = w_name.value
        existing_user.weight = w_weight.value
        existing_user.height = w_height.value
        existing_user.age = w_age.value
        existing_user.gender = w_gender.value
        existing_user.activity_level = w_activity.value
        existing_user.goal = w_goal.value
        existing_user.dietary_restrictions = w_restrictions.value
        
        # Commit changes to the database
        try:
            session.commit()
            print(f"SUCCESS! Profile for {existing_user.name} saved.")
            print(f"Goal: {existing_user.goal} | Weight: {existing_user.weight}kg")
        except Exception as e:
            session.rollback()
            print(f"Error saving to database: {e}")

btn_save.on_click(save_profile_to_db)

# --- Display the Form ---
display(widgets.VBox([
    widgets.HTML("<h3>User Profile Setup</h3>"),
    w_name, w_weight, w_height, w_age, w_gender,
    w_activity, w_goal, w_restrictions,
    btn_save, output
]))

In [8]:
import pandas as pd
from IPython.display import display

# Funkcja pomocnicza do wyświetlania tabeli
def show_table(model_class, table_name):
    # Pobierz zapytanie jako statement
    stmt = session.query(model_class).statement
    # Odczytaj SQL do DataFrame
    df = pd.read_sql(stmt, session.bind) #type: ignore
    
    print(f"--- ZAWARTOŚĆ TABELI: {table_name} ---")
    if not df.empty:
        display(df)
    else:
        print("(Tabela jest pusta)")
    print("\n")

# Wyświetl tabelę UserProfile
show_table(UserProfile, "User Profile")

# Wyświetl tabelę MealLog
show_table(MealLog, "Meal Log")

--- ZAWARTOŚĆ TABELI: User Profile ---


,id,name,weight,height,age,gender,activity_level,goal,dietary_restrictions
0,1,Adam,70.0,175.0,30,Male,sedentary,weight_loss,




--- ZAWARTOŚĆ TABELI: Meal Log ---
(Tabela jest pusta)




In [7]:
# db tests

from sqlalchemy import text

def run_system_health_check():
    print("STARTING SYSTEM TESTS...\n")
    
    # --- TEST 1: File Existence ---
    db_file = 'nowaste.db'
    if os.path.exists(db_file):
        print(f"PASS: Database file '{db_file}' exists.")
    else:
        print(f"FAIL: Database file '{db_file}' not found.")
        return # Stop tests if file is missing

    # --- TEST 2: Database Connection ---
    try:
        # Try to execute a simple SQL query
        session.execute(text("SELECT 1"))
        print("PASS: Database connection established.")
    except Exception as e:
        print(f"FAIL: Database connection failed. Error: {e}")
        return

    # --- TEST 3: CRUD Logic (Create, Read, Update, Delete) ---
    print("\nRunning CRUD Test (on temporary data)...")
    try:
        # 3.1 CREATE
        test_user = UserProfile(
            name="TestUnit_Ghost", 
            weight=100.0, 
            goal="test", 
            dietary_restrictions="none"
        )
        session.add(test_user)
        session.commit()
        
        # 3.2 READ
        retrieved_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert retrieved_user is not None, "Failed to retrieve created user"
        assert retrieved_user.weight == 100.0, "Weight mismatch" # type: ignore
        
        # 3.3 UPDATE
        retrieved_user.goal = "updated_goal" # type: ignore
        session.commit()
        updated_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert updated_user.goal == "updated_goal", "Update failed" # type: ignore
        
        # 3.4 DELETE (Cleanup)
        session.delete(updated_user)
        session.commit()
        deleted_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert deleted_user is None, "Delete failed"
        
        print("PASS: CRUD operations working correctly.")
        
    except AssertionError as ae:
        print(f"FAIL: Assertion failed: {ae}")
        session.rollback()
    except Exception as e:
        print(f"FAIL: CRUD Error: {e}")
        session.rollback()

    # --- TEST 4: Verify Your Real Profile ---
    print("\nVerifying Active User Profile...")
    real_user = session.query(UserProfile).first()
    if real_user:
        print(f"PASS: Active user found: '{real_user.name}'")
        print(f"\tDetails: {real_user.weight}kg, Goal: {real_user.goal}")
    else:
        print("WARNING: No active user profile found. (Did you click 'Save Profile' in the previous step?)")

    print("\nTESTS COMPLETED.")

# Run the tests
run_system_health_check()

STARTING SYSTEM TESTS...

PASS: Database file 'nowaste.db' exists.
PASS: Database connection established.

Running CRUD Test (on temporary data)...
PASS: CRUD operations working correctly.

Verifying Active User Profile...
PASS: Active user found: 'Adam'
	Details: 70.0kg, Goal: weight_loss

TESTS COMPLETED.
